In [1]:
# =====================
# 🔁 Imports & Setup
# =====================
import os
import cv2
import librosa
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import ASTModel, VideoMAEModel, HubertModel
from moviepy.editor import VideoFileClip
import glob
from sklearn.metrics import accuracy_score, confusion_matrix
from tqdm import tqdm

# =====================
# 📁 Dataset Paths
# =====================
BASE_PATH = "/kaggle/input/fakeav"
PROCESSED_PATH = "/kaggle/working/processed"
MEL_PATH = f"{PROCESSED_PATH}/audio_mels"
FRAME_PATH = f"{PROCESSED_PATH}/video_frames"
LIP_PATH = f"{PROCESSED_PATH}/lip_frames"
LABEL_CSV = f"{PROCESSED_PATH}/labels.csv"

os.makedirs(MEL_PATH, exist_ok=True)
os.makedirs(FRAME_PATH, exist_ok=True)
os.makedirs(LIP_PATH, exist_ok=True)

# =====================
# 📋 Dataset Organization
# =====================
def create_labels_csv():
    """Create labels CSV file based on directory structure"""
    data = []
    
    # Based on the input hierarchy in the screenshot
    categories = {
        "FakeVideo-FakeAudio": 0,  # Fake
        "FakeVideo-RealAudio": 0,  # Fake
        "RealVideo-FakeAudio": 0,  # Fake
        "RealVideo-RealAudio": 1   # Real
    }
    
    for category, label in categories.items():
        category_path = os.path.join(BASE_PATH, "FakeAVCeleb_v1.2", category)
        video_files = glob.glob(os.path.join(category_path, "*.mp4"))
        
        for video in video_files:
            video_id = os.path.basename(video).split('.')[0]
            data.append({
                "id": video_id,
                "path": video,
                "category": category,
                "label": label
            })
    
    df = pd.DataFrame(data)
    df.to_csv(LABEL_CSV, index=False)
    print(f"Created labels CSV with {len(df)} entries")
    return df

# =====================
# 🎧 Audio Processing
# =====================
def extract_audio(video_path, output_path):
    """Extract audio from video file"""
    clip = VideoFileClip(video_path)
    clip.audio.write_audiofile(output_path, fps=16000, verbose=False, logger=None)
    clip.close()

def generate_mel(audio_path, save_path):
    """Generate Mel spectrogram from audio file"""
    y, sr = librosa.load(audio_path, sr=16000)
    mel = librosa.feature.melspectrogram(y, sr=sr, n_mels=128)
    mel_db = librosa.power_to_db(mel, ref=np.max)
    np.save(save_path, mel_db)

def process_audio(video_path, video_id):
    """Process audio from video"""
    temp_audio = f"{PROCESSED_PATH}/{video_id}.wav"
    mel_save = f"{MEL_PATH}/{video_id}.npy"
    
    if not os.path.exists(mel_save):
        extract_audio(video_path, temp_audio)
        generate_mel(temp_audio, mel_save)
        if os.path.exists(temp_audio):
            os.remove(temp_audio)  # Clean up temp file

# =====================
# 🎥 Video Frame Extract
# =====================
def extract_frames(video_path, save_folder, skip=5):
    """Extract frames from video with skip interval"""
    cap = cv2.VideoCapture(video_path)
    count = 0
    saved = 0
    os.makedirs(save_folder, exist_ok=True)
    
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        if count % skip == 0:
            cv2.imwrite(f"{save_folder}/frame_{saved:04d}.jpg", frame)
            saved += 1
        count += 1
    
    cap.release()
    return saved

# =====================
# 👄 Lip Region Extract Using Face ROI
# =====================
def extract_face_roi(frames_folder, save_folder):
    """Extract face region of interest instead of exact lips due to missing landmarks model"""
    os.makedirs(save_folder, exist_ok=True)
    face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')
    
    frames = sorted(glob.glob(os.path.join(frames_folder, "*.jpg")))
    for img_path in frames:
        frame = cv2.imread(img_path)
        if frame is None:
            continue
            
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        faces = face_cascade.detectMultiScale(gray, 1.1, 4)
        
        # If face detected, extract lower third as approximate lip region
        if len(faces) > 0:
            x, y, w, h = faces[0]
            # Focus on lower part of face for approximate lip region
            lip_y = y + int(h * 0.6)
            lip_h = int(h * 0.4)
            lip_region = frame[lip_y:lip_y+lip_h, x:x+w]
            
            if lip_region.size > 0:  # Ensure valid crop
                lip_region = cv2.resize(lip_region, (112, 112))
                base_filename = os.path.basename(img_path)
                cv2.imwrite(os.path.join(save_folder, base_filename), lip_region)
        else:
            # If no face detected, just use center crop as fallback
            h, w = frame.shape[:2]
            center_y = h // 2
            crop = frame[center_y-56:center_y+56, w//2-56:w//2+56]
            if crop.size > 0 and crop.shape[0] > 0 and crop.shape[1] > 0:
                crop = cv2.resize(crop, (112, 112))
                base_filename = os.path.basename(img_path)
                cv2.imwrite(os.path.join(save_folder, base_filename), crop)

# =====================
# 🔄 Data Preprocessing
# =====================
def preprocess_data(df):
    """Process all videos in the dataset"""
    for _, row in tqdm(df.iterrows(), total=len(df)):
        video_id = row['id']
        video_path = row['path']
        
        # Process audio
        process_audio(video_path, video_id)
        
        # Process video frames
        frames_folder = os.path.join(FRAME_PATH, video_id)
        if not os.path.exists(frames_folder) or len(os.listdir(frames_folder)) == 0:
            extract_frames(video_path, frames_folder, skip=3)
        
        # Process lip regions
        lip_folder = os.path.join(LIP_PATH, video_id)
        if not os.path.exists(lip_folder) or len(os.listdir(lip_folder)) == 0:
            extract_face_roi(frames_folder, lip_folder)

# =====================
# 📦 Dataset Classes
# =====================
class AudioDataset(Dataset):
    def __init__(self, csv, mel_dir, transform=None):
        self.data = pd.read_csv(csv)
        self.mel_dir = mel_dir
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        mel_path = os.path.join(self.mel_dir, f"{row['id']}.npy")
        
        try:
            mel = np.load(mel_path)
            if self.transform:
                mel = self.transform(mel)
            # Ensure mel has right shape and is padded/truncated if needed
            mel = self._prepare_mel(mel)
            return {"input": torch.tensor(mel).float(), "label": torch.tensor(row['label'])}
        except Exception as e:
            print(f"Error loading {mel_path}: {e}")
            # Return a zero tensor with proper shape as fallback
            return {"input": torch.zeros((1, 128, 1000)).float(), "label": torch.tensor(row['label'])}
    
    def _prepare_mel(self, mel):
        """Ensure mel spectrogram has consistent size"""
        target_len = 1000  # Target time dimension
        
        # Add channel dimension if needed
        if len(mel.shape) == 2:
            mel = np.expand_dims(mel, axis=0)
            
        # Resize time dimension
        curr_len = mel.shape[2] if len(mel.shape) == 3 else mel.shape[1]
        if curr_len < target_len:
            # Pad
            pad_len = target_len - curr_len
            if len(mel.shape) == 3:
                mel = np.pad(mel, ((0, 0), (0, 0), (0, pad_len)), mode='constant')
            else:
                mel = np.pad(mel, ((0, 0), (0, pad_len)), mode='constant')
        elif curr_len > target_len:
            # Truncate
            if len(mel.shape) == 3:
                mel = mel[:, :, :target_len]
            else:
                mel = mel[:, :target_len]
                
        return mel

class VideoDataset(Dataset):
    def __init__(self, csv, frame_dir, num_frames=16):
        self.data = pd.read_csv(csv)
        self.frame_dir = frame_dir
        self.num_frames = num_frames

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        folder = os.path.join(self.frame_dir, row['id'])
        
        try:
            files = sorted(os.listdir(folder))
            if len(files) >= self.num_frames:
                files = files[:self.num_frames]
            else:
                # If we don't have enough frames, duplicate the last one
                files = files + [files[-1]] * (self.num_frames - len(files))
            
            imgs = []
            for f in files:
                img = cv2.imread(os.path.join(folder, f))
                if img is None:
                    # Create a blank image if file is corrupt
                    img = np.zeros((224, 224, 3), dtype=np.uint8)
                else:
                    img = cv2.resize(img, (224, 224))
                imgs.append(img)
            
            stack = np.stack(imgs).transpose(0, 3, 1, 2)
            return {"input": torch.tensor(stack / 255.0).float(), "label": torch.tensor(row['label'])}
        except Exception as e:
            print(f"Error loading video frames for {row['id']}: {e}")
            # Return a zero tensor with proper shape as fallback
            return {"input": torch.zeros((self.num_frames, 3, 224, 224)).float(), "label": torch.tensor(row['label'])}

class AVDataset(Dataset):
    def __init__(self, csv, lip_dir, mel_dir):
        self.data = pd.read_csv(csv)
        self.lip_dir = lip_dir
        self.mel_dir = mel_dir

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        lip_folder = os.path.join(self.lip_dir, row['id'])
        
        try:
            # Process lip frames
            if os.path.exists(lip_folder) and len(os.listdir(lip_folder)) > 0:
                lip_imgs = sorted(os.listdir(lip_folder))
                if len(lip_imgs) >= 16:
                    lip_imgs = lip_imgs[:16]
                else:
                    # Pad with duplicates of last frame if needed
                    lip_imgs = lip_imgs + [lip_imgs[-1]] * (16 - len(lip_imgs))
                    
                lips = []
                for f in lip_imgs:
                    img = cv2.imread(os.path.join(lip_folder, f))
                    if img is None:
                        img = np.zeros((112, 112, 3), dtype=np.uint8)
                    else:
                        img = cv2.resize(img, (112, 112))
                    lips.append(img)
                lips_stack = np.stack(lips).transpose(0, 3, 1, 2)
            else:
                # Create empty tensor if no lip frames available
                lips_stack = np.zeros((16, 3, 112, 112), dtype=np.float32)
            
            # Process audio mel
            mel_path = os.path.join(self.mel_dir, f"{row['id']}.npy")
            if os.path.exists(mel_path):
                mel = np.load(mel_path)
                # Ensure consistent shape
                if len(mel.shape) == 2:
                    mel = np.expand_dims(mel, axis=0) 
                
                # Pad or truncate to fixed length
                target_len = 1000
                curr_len = mel.shape[2] if len(mel.shape) == 3 else mel.shape[1]
                if curr_len < target_len:
                    pad_len = target_len - curr_len
                    mel = np.pad(mel, ((0, 0), (0, 0), (0, pad_len)), mode='constant')
                elif curr_len > target_len:
                    mel = mel[:, :, :target_len]
            else:
                mel = np.zeros((1, 128, 1000), dtype=np.float32)
            
            return {
                "input": {
                    "audio": torch.tensor(mel).float(), 
                    "lips": torch.tensor(lips_stack / 255.0).float()
                },
                "label": torch.tensor(row['label'])
            }
        except Exception as e:
            print(f"Error processing AV data for {row['id']}: {e}")
            return {
                "input": {
                    "audio": torch.zeros((1, 128, 1000)).float(),
                    "lips": torch.zeros((16, 3, 112, 112)).float()
                },
                "label": torch.tensor(row['label'])
            }

# =====================
# 🤖 Models
# =====================
class AudioModel(nn.Module):
    def __init__(self, pretrained=True):
        super().__init__()
        
        # Use a CNN-based approach instead of AST for more efficient training
        self.conv1 = nn.Conv2d(1, 64, kernel_size=3, stride=1, padding=1)
        self.bn1 = nn.BatchNorm2d(64)
        self.relu = nn.ReLU()
        self.pool = nn.MaxPool2d(2)
        
        self.conv2 = nn.Conv2d(64, 128, kernel_size=3, stride=1, padding=1)
        self.bn2 = nn.BatchNorm2d(128)
        
        self.conv3 = nn.Conv2d(128, 256, kernel_size=3, stride=1, padding=1)
        self.bn3 = nn.BatchNorm2d(256)
        
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(256, 2)
        
    def forward(self, x):
        # Input shape: [batch_size, 1, freq_bins, time]
        x = self.pool(self.relu(self.bn1(self.conv1(x))))
        x = self.pool(self.relu(self.bn2(self.conv2(x))))
        x = self.pool(self.relu(self.bn3(self.conv3(x))))
        
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        return self.fc(x)

class VideoModel(nn.Module):
    def __init__(self, pretrained=True):
        super().__init__()
        
        # Use a 3D CNN approach instead of full VideoMAE
        self.conv1 = nn.Conv3d(3, 64, kernel_size=(3, 7, 7), stride=(1, 2, 2), padding=(1, 3, 3))
        self.bn1 = nn.BatchNorm3d(64)
        self.relu = nn.ReLU()
        self.pool1 = nn.MaxPool3d(kernel_size=(1, 3, 3), stride=(1, 2, 2), padding=(0, 1, 1))
        
        self.conv2 = nn.Conv3d(64, 128, kernel_size=3, stride=1, padding=1)
        self.bn2 = nn.BatchNorm3d(128)
        self.pool2 = nn.MaxPool3d(kernel_size=2)
        
        self.conv3 = nn.Conv3d(128, 256, kernel_size=3, stride=1, padding=1)
        self.bn3 = nn.BatchNorm3d(256)
        self.pool3 = nn.MaxPool3d(kernel_size=2)
        
        # Global average pooling and classification
        self.avgpool = nn.AdaptiveAvgPool3d((1, 1, 1))
        self.fc = nn.Linear(256, 2)
        
    def forward(self, x):
        # Input shape: [batch_size, time_frames, channels, height, width]
        x = self.pool1(self.relu(self.bn1(self.conv1(x))))
        x = self.pool2(self.relu(self.bn2(self.conv2(x))))
        x = self.pool3(self.relu(self.bn3(self.conv3(x))))
        
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        return self.fc(x)

class AVModel(nn.Module):
    def __init__(self):
        super().__init__()
        # Audio encoder - simplified
        self.audio_encoder = nn.Sequential(
            nn.Conv2d(1, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.AdaptiveAvgPool2d((1, 1))
        )
        
        # Visual encoder - simplified 3D CNN
        self.visual_encoder = nn.Sequential(
            nn.Conv3d(3, 32, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm3d(32),
            nn.ReLU(),
            nn.MaxPool3d(kernel_size=2),
            nn.Conv3d(32, 64, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm3d(64),
            nn.ReLU(),
            nn.MaxPool3d(kernel_size=2),
            nn.AdaptiveAvgPool3d((1, 1, 1))
        )
        
        # Fusion and classification
        self.fc = nn.Sequential(
            nn.Linear(128 + 64, 128),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(128, 2)
        )

    def forward(self, input_dict):
        # Process audio
        audio_input = input_dict['audio']
        audio_feat = self.audio_encoder(audio_input)
        audio_feat = torch.flatten(audio_feat, 1)
        
        # Process lip video
        lip_input = input_dict['lips']
        lip_feat = self.visual_encoder(lip_input)
        lip_feat = torch.flatten(lip_feat, 1)
        
        # Combine features
        combined = torch.cat((audio_feat, lip_feat), dim=1)
        return self.fc(combined)

# =====================
# 🤝 Fusion Function
# =====================
def majority_vote(p1, p2, p3):
    return ((p1 + p2 + p3) >= 2).long()

def weighted_avg_score(log1, log2, log3, weights=[0.4, 0.3, 0.3]):
    """Weighted average of model outputs with customizable weights"""
    w1, w2, w3 = weights
    avg = (log1 * w1 + log2 * w2 + log3 * w3)
    return torch.argmax(avg, dim=1)

# =====================
# 🔁 Training & Eval
# =====================
def train(model, dataloader, optimizer, criterion, device, epochs=5):
    """Train model for specified epochs"""
    best_loss = float('inf')
    history = {'train_loss': [], 'val_loss': [], 'val_acc': []}
    
    for epoch in range(epochs):
        model.train()
        total_loss = 0
        
        progress_bar = tqdm(dataloader, desc=f"Epoch {epoch+1}/{epochs}")
        for batch in progress_bar:
            # Handle different input types for AV model vs single-modality models
            if isinstance(batch['input'], dict):
                # For AV model
                inputs = {k: v.to(device) for k, v in batch['input'].items()}
            else:
                # For Audio-only or Video-only models
                inputs = batch['input'].to(device)
                
            labels = batch['label'].to(device)
            
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            total_loss += loss.item()
            progress_bar.set_postfix({'loss': loss.item()})
            
        avg_loss = total_loss / len(dataloader)
        history['train_loss'].append(avg_loss)
        print(f"Epoch {epoch+1}/{epochs}, Loss: {avg_loss:.4f}")
        
        # Save best model
        if avg_loss < best_loss:
            best_loss = avg_loss
            torch.save(model.state_dict(), f"{PROCESSED_PATH}/best_model.pt")
    
    return history

def evaluate(model, dataloader, device):
    """Evaluate model performance"""
    model.eval()
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Evaluating"):
            # Handle different input types for AV model vs single-modality models
            if isinstance(batch['input'], dict):
                # For AV model
                inputs = {k: v.to(device) for k, v in batch['input'].items()}
            else:
                # For Audio-only or Video-only models
                inputs = batch['input'].to(device)
                
            labels = batch['label'].to(device)
            
            outputs = model(inputs)
            preds = torch.argmax(outputs, dim=1)
            
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    acc = accuracy_score(all_labels, all_preds)
    conf_mat = confusion_matrix(all_labels, all_preds)
    
    print(f"Accuracy: {acc:.4f}")
    print("Confusion Matrix:")
    print(conf_mat)
    
    return acc, conf_mat

# =====================
# 🚀 Main Pipeline
# =====================
def run_pipeline():
    """Run full AVTENet pipeline"""
    # 1. Create labels CSV from dataset structure
    if not os.path.exists(LABEL_CSV):
        labels_df = create_labels_csv()
    else:
        labels_df = pd.read_csv(LABEL_CSV)
    
    print(f"Dataset contains {len(labels_df)} videos with class distribution:")
    print(labels_df['label'].value_counts())
    
    # 2. Preprocess data (audio, video frames, lip regions)
    preprocess_data(labels_df)
    
    # 3. Split data into train/validation sets
    from sklearn.model_selection import train_test_split
    train_df, val_df = train_test_split(labels_df, test_size=0.2, stratify=labels_df['label'], random_state=42)
    
    train_df.to_csv(f"{PROCESSED_PATH}/train.csv", index=False)
    val_df.to_csv(f"{PROCESSED_PATH}/val.csv", index=False)
    
    print(f"Training set: {len(train_df)} videos")
    print(f"Validation set: {len(val_df)} videos")
    
    # 4. Create datasets and dataloaders
    batch_size = 8
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")
    
    # Audio
    train_audio_dataset = AudioDataset(f"{PROCESSED_PATH}/train.csv", MEL_PATH)
    val_audio_dataset = AudioDataset(f"{PROCESSED_PATH}/val.csv", MEL_PATH)
    
    train_audio_loader = DataLoader(train_audio_dataset, batch_size=batch_size, shuffle=True)
    val_audio_loader = DataLoader(val_audio_dataset, batch_size=batch_size)
    
    # Video
    train_video_dataset = VideoDataset(f"{PROCESSED_PATH}/train.csv", FRAME_PATH)
    val_video_dataset = VideoDataset(f"{PROCESSED_PATH}/val.csv", FRAME_PATH)
    
    train_video_loader = DataLoader(train_video_dataset, batch_size=batch_size, shuffle=True)
    val_video_loader = DataLoader(val_video_dataset, batch_size=batch_size)
    
    # Audio-Visual
    train_av_dataset = AVDataset(f"{PROCESSED_PATH}/train.csv", LIP_PATH, MEL_PATH)
    val_av_dataset = AVDataset(f"{PROCESSED_PATH}/val.csv", LIP_PATH, MEL_PATH)
    
    train_av_loader = DataLoader(train_av_dataset, batch_size=batch_size, shuffle=True)
    val_av_loader = DataLoader(val_av_dataset, batch_size=batch_size)
    
    # 5. Initialize models
    audio_model = AudioModel().to(device)
    video_model = VideoModel().to(device)
    av_model = AVModel().to(device)
    
    criterion = nn.CrossEntropyLoss()
    
    # 6. Train Audio Model
    print("\n===== Training Audio Model =====")
    audio_optimizer = torch.optim.Adam(audio_model.parameters(), lr=1e-4)
    audio_history = train(audio_model, train_audio_loader, audio_optimizer, criterion, device, epochs=5)
    
    # 7. Evaluate Audio Model
    print("\n===== Evaluating Audio Model =====")
    audio_acc, audio_conf = evaluate(audio_model, val_audio_loader, device)
    
    # 8. Train Video Model
    print("\n===== Training Video Model =====")
    video_optimizer = torch.optim.Adam(video_model.parameters(), lr=1e-4)
    video_history = train(video_model, train_video_loader, video_optimizer, criterion, device, epochs=5)
    
    # 9. Evaluate Video Model
    print("\n===== Evaluating Video Model =====")
    video_acc, video_conf = evaluate(video_model, val_video_loader, device)
    
    # 10. Train AV Model
    print("\n===== Training Audio-Visual Model =====")
    av_optimizer = torch.optim.Adam(av_model.parameters(), lr=1e-4)
    av_history = train(av_model, train_av_loader, av_optimizer, criterion, device, epochs=5)
    
    # 11. Evaluate AV Model
    print("\n===== Evaluating Audio-Visual Model =====")
    av_acc, av_conf = evaluate(av_model, val_av_loader, device)
    
    # 12. Ensemble Prediction (sample code for inference)
    def ensemble_predict(audio_path, video_path):
        """Sample ensemble prediction function"""
        # Preprocess audio
        audio_id = "temp_audio"
        process_audio(video_path, audio_id)
        
        # Preprocess video
        video_id = "temp_video"
        frames_folder = os.path.join(FRAME_PATH, video_id)
        extract_frames(video_path, frames_folder)
        
        lip_folder = os.path.join(LIP_PATH, video_id)
        extract_face_roi(frames_folder, lip_folder)
        
        # Load preprocessed data
        mel = np.load(f"{MEL_PATH}/{audio_id}.npy")
        mel = np.expand_dims(mel, axis=0)
        
        # Prepare frames
        frame_files = sorted(os.listdir(frames_folder))[:16]
        frames = [cv2.resize(cv2.imread(os.path.join(frames_folder, f)), (224, 224)) for f in frame_files]
        frames_stack = np.stack(frames).transpose(0, 3, 1, 2) / 255.0
        
        # Prepare lip frames
        lip_files = sorted(os.listdir(lip_folder))[:16]
        lips = [cv2.resize(cv2.imread(os.path.join(lip_folder, f)), (112, 112)) for f in lip_files]
        lips_stack = np.stack(lips).transpose(0, 3, 1, 2) / 255.0
        
        # Model predictions
        with torch.no_grad():
            # Audio prediction
            audio_input = torch.tensor(mel).float().to(device)
            audio_output = audio_model(audio_input)
            audio_pred = torch.softmax(audio_output, dim=1)
            
            # Video prediction
            video_input = torch.tensor(frames_stack).float().to(device)
            video_output = video_model(video_input)
            video_pred = torch.softmax(video_output, dim=1)
            
            # AV prediction
            av_input = {
                'audio': torch.tensor(mel).float().to(device),
                 'lips': torch.tensor(lips_stack).float().to(device)
            }
            av_output = av_model(av_input)
            av_pred = torch.softmax(av_output, dim=1)
            
            # Ensemble decision using weighted average
            final_pred = weighted_avg_score(
                audio_pred, 
                video_pred, 
                av_pred, 
                weights=[0.3, 0.3, 0.4]
            )
            
            # Convert to label
            result = "Real" if final_pred.item() == 1 else "Fake"
            
        # Clean up temp files
        for path in [f"{MEL_PATH}/{audio_id}.npy", frames_folder, lip_folder]:
            if os.path.exists(path):
                if os.path.isdir(path):
                    import shutil
                    shutil.rmtree(path)
                else:
                    os.remove(path)
                    
        return result, {
            'audio_confidence': audio_pred.cpu().numpy(),
            'video_confidence': video_pred.cpu().numpy(),
            'av_confidence': av_pred.cpu().numpy()
        }
    
    # 13. Save Results and Models
    results = {
        'audio_accuracy': audio_acc,
        'video_accuracy': video_acc, 
        'av_accuracy': av_acc,
        'ensemble_weight_audio': 0.3,
        'ensemble_weight_video': 0.3,
        'ensemble_weight_av': 0.4
    }
    
    # Save results to JSON
    import json
    with open(f"{PROCESSED_PATH}/model_results.json", "w") as f:
        json.dump(results, f)
    
    # Save models
    torch.save(audio_model.state_dict(), f"{PROCESSED_PATH}/audio_model.pt")
    torch.save(video_model.state_dict(), f"{PROCESSED_PATH}/video_model.pt")
    torch.save(av_model.state_dict(), f"{PROCESSED_PATH}/av_model.pt")
    
    print("\n===== Models and Results Saved =====")
    print(f"Audio Model Accuracy: {audio_acc:.4f}")
    print(f"Video Model Accuracy: {video_acc:.4f}")
    print(f"Audio-Visual Model Accuracy: {av_acc:.4f}")
    
    return audio_model, video_model, av_model, results

# =====================
# 🧪 Demo Function
# =====================
def demo_detection(audio_model, video_model, av_model, video_path):
    """Run detection on a sample video"""
    # Set models to evaluation mode
    audio_model.eval()
    video_model.eval()
    av_model.eval()
    
    # Extract filename for display
    video_name = os.path.basename(video_path)
    print(f"\n===== Running detection on {video_name} =====")
    
    # Run prediction
    result, confidences = ensemble_predict(video_path, video_path)
    
    # Display results
    print(f"Final verdict: {result}")
    print(f"Audio model confidence: Real={confidences['audio_confidence'][0][1]:.2f}, Fake={confidences['audio_confidence'][0][0]:.2f}")
    print(f"Video model confidence: Real={confidences['video_confidence'][0][1]:.2f}, Fake={confidences['video_confidence'][0][0]:.2f}")
    print(f"AV model confidence: Real={confidences['av_confidence'][0][1]:.2f}, Fake={confidences['av_confidence'][0][0]:.2f}")
    
    return result, confidences

# =====================
# 📊 Visualization
# =====================
def plot_results(results):
    """Plot model accuracy comparison"""
    import matplotlib.pyplot as plt
    
    # Bar plot of accuracies
    models = ['Audio Model', 'Video Model', 'Audio-Visual Model']
    accuracies = [results['audio_accuracy'], results['video_accuracy'], results['av_accuracy']]
    
    plt.figure(figsize=(10, 6))
    bars = plt.bar(models, accuracies, color=['#3498db', '#2ecc71', '#e74c3c'])
    
    plt.ylim(0, 1.0)
    plt.title('Model Accuracy Comparison', fontsize=16)
    plt.ylabel('Accuracy', fontsize=14)
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    
    # Add value labels on top of bars
    for bar in bars:
        height = bar.get_height()
        plt.text(bar.get_x() + bar.get_width()/2., height + 0.01,
                 f'{height:.4f}', ha='center', fontsize=12)
    
    plt.tight_layout()
    plt.savefig(f"{PROCESSED_PATH}/accuracy_comparison.png")
    plt.show()
    
    return plt

# =====================
# 🔍 Error Analysis
# =====================
def analyze_errors(model, dataloader, device, name="model"):
    """Analyze which samples the model gets wrong"""
    model.eval()
    errors = []
    
    with torch.no_grad():
        for batch in tqdm(dataloader, desc=f"Analyzing {name} errors"):
            # Handle different input types
            if isinstance(batch['input'], dict):
                inputs = {k: v.to(device) for k, v in batch['input'].items()}
            else:
                inputs = batch['input'].to(device)
                
            labels = batch['label'].cpu().numpy()
            ids = batch['id'] if 'id' in batch else [f"sample_{i}" for i in range(len(labels))]
            
            outputs = model(inputs)
            preds = torch.argmax(outputs, dim=1).cpu().numpy()
            
            # Track errors
            for i, (pred, label) in enumerate(zip(preds, labels)):
                if pred != label:
                    errors.append({
                        'id': ids[i],
                        'predicted': "Real" if pred == 1 else "Fake",
                        'actual': "Real" if label == 1 else "Fake",
                        'confidence': torch.softmax(outputs, dim=1)[i][pred].item()
                    })
    
    print(f"\n{name} made {len(errors)} errors")
    if errors:
        # Display a few examples
        print("Sample errors:")
        for i, err in enumerate(errors[:5]):
            print(f"{i+1}. ID: {err['id']}, Predicted: {err['predicted']}, Actual: {err['actual']}, Confidence: {err['confidence']:.4f}")
    
    # Save errors to CSV
    error_df = pd.DataFrame(errors)
    if not error_df.empty:
        error_df.to_csv(f"{PROCESSED_PATH}/{name}_errors.csv", index=False)
    
    return errors

# =====================
# 🏃‍♂️ Runner
# =====================
if __name__ == "__main__":
    print("Starting AVTENet Deepfake Detection Pipeline")
    
    # Run full pipeline
    audio_model, video_model, av_model, results = run_pipeline()
    
    # Plot results
    plot_results(results)
    
    # If test samples are available, run demo
    test_samples = glob.glob(os.path.join(BASE_PATH, "test_samples", "*.mp4"))
    if test_samples:
        for sample in test_samples[:3]:  # Test on first 3 samples
            demo_detection(audio_model, video_model, av_model, sample)
    
    # Error analysis on validation data
    val_df = pd.read_csv(f"{PROCESSED_PATH}/val.csv")
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    # Create datasets with IDs for error analysis
    class IDDataset(AudioDataset):
        def __getitem__(self, idx):
            item = super().__getitem__(idx)
            item['id'] = self.data.iloc[idx]['id']
            return item
    
    val_audio_dataset = IDDataset(f"{PROCESSED_PATH}/val.csv", MEL_PATH)
    val_audio_loader = DataLoader(val_audio_dataset, batch_size=8)
    
    print("\n===== Error Analysis =====")
    analyze_errors(audio_model, val_audio_loader, device, name="audio_model")
    
    print("\nAVTENet Pipeline Complete!")

2025-04-17 08:18:13.802499: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1744877893.998341      31 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1744877894.055899      31 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
error: XDG_RUNTIME_DIR not set in the environment.
ALSA lib confmisc.c:855:(parse_card) cannot find card '0'
ALSA lib conf.c:5178:(_snd_config_evaluate) function snd_func_card_inum returned error: No such file or directory
ALSA lib confmisc.c:422:(snd_func_concat) error evaluating strings
ALSA lib conf.c:5178:(_snd_config_evaluate) function snd_func_concat returned error: No such file or directory
ALSA lib confmisc.c:1334:(snd_func_r

Starting AVTENet Deepfake Detection Pipeline
Created labels CSV with 0 entries
Dataset contains 0 videos with class distribution:


KeyError: 'label'